In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SEGMENTASI = ROOT / "data" / "segmentasi"   # objek berwarna, latar hitam (dari notebook 03)
MASK = ROOT / "data" / "mask"               # mask objek (dari notebook 03)
HASIL = ROOT / "hasil" / "konvolusi"

KELAS = ["logam", "paper", "plastic"]
JUMLAH_CONTOH = 2

# Kernel konvolusi 3x3
KERNEL_SOBEL_X = np.array([[-1, 0, 1],
                           [-2, 0, 2],
                           [-1, 0, 1]], dtype=np.float32)

KERNEL_SOBEL_Y = np.array([[-1, -2, -1],
                           [ 0,  0,  0],
                           [ 1,  2,  1]], dtype=np.float32)

KERNEL_LAPLACIAN = np.array([[ 0,  1,  0],
                             [ 1, -4,  1],
                             [ 0,  1,  0]], dtype=np.float32)

def baca(path, gray=False):
    """Membaca citra dengan aman untuk path Windows."""
    try:
        data = np.fromfile(str(path), np.uint8)
        flag = cv2.IMREAD_GRAYSCALE if gray else cv2.IMREAD_COLOR
        return cv2.imdecode(data, flag)
    except Exception:
        return None

def simpan(path, img):
    """Menyimpan citra dengan aman untuk path Windows."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, buffer = cv2.imencode(path.suffix, img)
    if ok:
        buffer.tofile(str(path))

print("Folder segmentasi :", SEGMENTASI)
print("Folder mask       :", MASK)
print("Folder output     :", HASIL)

## 2. Fungsi konvolusi



In [ ]:
def terapkan_konvolusi(gray, kernel):
    """Menerapkan kernel 3x3 secara eksplisit dengan OpenCV filter2D."""
    return cv2.filter2D(
        gray.astype(np.float32),
        ddepth=cv2.CV_32F,
        kernel=kernel,
        borderType=cv2.BORDER_REFLECT
    )

def normalisasi_tampilan(feature_map):
    """Normalisasi hanya untuk menyimpan/menampilkan feature map."""
    nilai = np.abs(feature_map)
    maksimum = nilai.max()
    if maksimum == 0:
        return np.zeros(nilai.shape, dtype=np.uint8)
    return np.clip(nilai / maksimum * 255, 0, 255).astype(np.uint8)

def hitung_ringkasan(feature_map, mask_objek):
    """Statistik respons konvolusi pada piksel objek saja."""
    nilai = feature_map[mask_objek].astype(np.float64)
    if nilai.size == 0:
        return None
    return {
        "rata_abs": float(np.mean(np.abs(nilai))),
        "std": float(np.std(nilai)),
        "energi": float(np.mean(nilai ** 2))
    }

def proses_konvolusi(citra_warna, mask):
    gray = cv2.cvtColor(citra_warna, cv2.COLOR_BGR2GRAY)
    objek = mask > 0

    sx = terapkan_konvolusi(gray, KERNEL_SOBEL_X)
    sy = terapkan_konvolusi(gray, KERNEL_SOBEL_Y)
    lap = terapkan_konvolusi(gray, KERNEL_LAPLACIAN)

    # Magnitudo gradien gabungan Sobel X dan Sobel Y
    magnitude = np.sqrt(sx ** 2 + sy ** 2)

    # Nolkan respons di luar objek
    for fmap in (sx, sy, lap, magnitude):
        fmap[~objek] = 0

    return gray, {
        "sobel_x": sx,
        "sobel_y": sy,
        "laplacian": lap,
        "gradien_magnitude": magnitude
    }

## 3. Terapkan konvolusi pada seluruh citra hasil segmentasi



In [ ]:
baris_fitur = []
dilewati = []

for kelas in KELAS:
    folder = SEGMENTASI / kelas
    if not folder.exists():
        print(f"Folder tidak ditemukan: {folder}")
        continue

    for path in sorted(folder.glob("*.png")):
        citra = baca(path)
        mask = baca(MASK / kelas / path.name, gray=True)

        if citra is None or mask is None or not np.any(mask > 0):
            dilewati.append(f"{kelas}/{path.name}")
            continue

        gray, maps = proses_konvolusi(citra, mask)
        objek = mask > 0

        fitur = {"nama": path.name, "kelas": kelas}
        for nama_map, fmap in maps.items():
            statistik = hitung_ringkasan(fmap, objek)
            if statistik is None:
                continue
            for nama_stat, nilai in statistik.items():
                fitur[f"{nama_map}_{nama_stat}"] = nilai

            # Simpan feature map versi visualisasi
            tujuan = HASIL / "feature_map" / nama_map / kelas / path.name
            simpan(tujuan, normalisasi_tampilan(fmap))

        baris_fitur.append(fitur)

fitur_konvolusi = pd.DataFrame(baris_fitur)
HASIL.mkdir(parents=True, exist_ok=True)
fitur_konvolusi.to_csv(HASIL / "fitur_konvolusi.csv", index=False)

print("Jumlah citra berhasil diproses:", len(fitur_konvolusi))
print("Jumlah citra dilewati:", len(dilewati))
if dilewati:
    print("Contoh file yang dilewati:", dilewati[:10])

display(fitur_konvolusi.head().round(4))

## 4. Visualisasi hasil konvolusi



In [ ]:
contoh = []
for kelas in KELAS:
    data_kelas = fitur_konvolusi[fitur_konvolusi["kelas"] == kelas]
    if not data_kelas.empty:
        contoh.extend(data_kelas.head(JUMLAH_CONTOH).to_dict("records"))

for item in contoh:
    nama = item["nama"]
    kelas = item["kelas"]
    citra = baca(SEGMENTASI / kelas / nama)
    mask = baca(MASK / kelas / nama, gray=True)
    gray, maps = proses_konvolusi(citra, mask)

    fig, ax = plt.subplots(1, 6, figsize=(18, 4))
    ax[0].imshow(cv2.cvtColor(citra, cv2.COLOR_BGR2RGB))
    ax[0].set_title("Segmentasi berwarna")
    ax[1].imshow(gray, cmap="gray")
    ax[1].set_title("Grayscale")

    for a, (nama_map, fmap) in zip(ax[2:], maps.items()):
        a.imshow(normalisasi_tampilan(fmap), cmap="gray")
        a.set_title(nama_map.replace("_", " ").title())

    for a in ax:
        a.axis("off")
    fig.suptitle(f"Hasil konvolusi: {kelas} - {nama}")
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## 5. Ringkasan fitur konvolusi per kelas



In [ ]:
if not fitur_konvolusi.empty:
    kolom_numerik = fitur_konvolusi.select_dtypes(include="number").columns
    display(
        fitur_konvolusi.groupby("kelas")[kolom_numerik]
        .mean()
        .round(4)
    )
else:
    print("Belum ada fitur. Pastikan notebook 03 sudah dijalankan dan folder data/segmentasi serta data/mask tersedia.")